# 18.5 概率编程与层次模型 / Probabilistic Programming & Hierarchical Models

**中文**：前面几节我们**手写**了每个模型的推断(共轭闭式、Laplace、MCMC、变分)。但真实建模里,你不想每换一个模型就重推一遍数学、重写一遍采样器。**概率编程语言(Probabilistic Programming Language, PPL)**——如 **PyMC、Stan、NumPyro**——解决了这个痛点:*"你只需用几行代码**声明生成模型**(先验 + 似然),PPL 就自动帮你做推断(自动 MCMC/变分)。"* 它把**建模**和**推断**彻底解耦。本节以 PPL 的杀手级应用——**层次模型(hierarchical model)** 为例,从零实现它,并说明 PPL 在背后替你做了什么。
**English**: Prior sections **hand-coded** inference for each model (conjugate closed form, Laplace, MCMC, VI). But in real modeling you don't want to re-derive the math and rewrite a sampler for every new model. **Probabilistic Programming Languages (PPLs)** — PyMC, Stan, NumPyro — solve this: *"you just **declare the generative model** (priors + likelihood) in a few lines, and the PPL automatically does inference (auto MCMC/VI)."* They fully decouple **modeling** from **inference**. This section uses PPLs' killer app — **hierarchical models** — implementing it from scratch and explaining what the PPL does for you.

---

**中文**：**层次(多层)模型**解决一个极常见的问题:估计**很多相关小组**各自的指标,而每组数据都不多。典型场景:
**English**: **Hierarchical (multilevel) models** solve a very common problem: estimating a metric for **many related groups**, each with little data. Typical scenarios:
- 每个**棒球运动员**的真实打击率(有人只上场 10 次,有人 500 次)。
  Each **baseball player**'s true batting average (some batted 10 times, others 500).
- 每个**门店/地区/用户群**的转化率;每个**学校**的教学效果。
  Each **store/region/user-segment**'s conversion rate; each **school**'s teaching effect.

**中文**：有三种做法:
**English**: Three approaches:
- **不合并(no pooling)**:每组各算各的(如打击率 = 命中/上场)。**小样本组噪声极大**(上场 3 次全中 → 估计 100%？荒谬)。
  **No pooling**: estimate each group alone (e.g. hits/at-bats). **Tiny-sample groups are wildly noisy** (3-for-3 → 100%? absurd).
- **完全合并(complete pooling)**:所有组用同一个全局平均。**抹掉了组间真实差异**。
  **Complete pooling**: everyone gets the same global average. **Erases real between-group differences**.
- **部分合并(partial pooling = 层次模型)**:每组估计**向全局均值"收缩(shrink)"**,收缩多少取决于该组数据量——**数据少的组多收缩(信全局)、数据多的组少收缩(信自己)**。这是"两个极端之间的最优折中",数学上由层次结构自动实现。

  **Partial pooling (= hierarchical model)**: each group's estimate **"shrinks" toward the global mean**, by an amount depending on its sample size — **small-sample groups shrink more (trust the global), large-sample groups shrink less (trust their own data)**. The optimal compromise between the two extremes, achieved automatically by the hierarchy.

**中文**：层次的意思是:各组参数 $\theta_k$ **不是独立的**,而是**从一个共享的"母分布"里抽出来的**——$\theta_k\sim p(\theta|\phi)$,母分布的超参数 $\phi$ 也要从所有组的数据里一起估。于是各组通过母分布**互相"借力(borrow strength)"**。
**English**: "Hierarchical" means: group parameters $\theta_k$ are **not independent** but **drawn from a shared "parent" distribution** — $\theta_k\sim p(\theta|\phi)$, whose hyperparameters $\phi$ are estimated jointly from all groups' data. So groups **"borrow strength"** from one another via the parent.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 层次模型必考）**
> **中文**：**概率编程(PyMC/Stan/NumPyro)**=声明生成模型(先验+似然), 自动推断(NUTS/ADVI), 建模与推断解耦。**层次模型**=各组参数共享一个母分布→**部分合并/收缩(shrinkage)**:小样本组向全局均值收缩多、大样本组收缩少。**为什么赢**:在"每组各算(高方差)"和"全合并(高偏差)"之间取最优偏差-方差折中——**Stein 悖论/经验贝叶斯**的实践形态, 常比两个极端都准。**关键结构**:$y_k\sim \text{Lik}(\theta_k)$, $\theta_k\sim p(\theta|\phi)$, $\phi\sim$ 超先验。用途:A/B 跨细分、多地区/多门店、推荐冷启动、meta 分析。诚实点:超先验选择、$\tau$(组间方差)的推断在数据少时敏感(漏斗问题→需重参数化)。
> **English**: **Probabilistic programming (PyMC/Stan/NumPyro)** = declare the generative model (priors + likelihood), auto-inference (NUTS/ADVI), decoupling modeling from inference. **Hierarchical model** = group parameters share a parent distribution → **partial pooling / shrinkage**: small-sample groups shrink more toward the global mean, large-sample groups less. **Why it wins**: the optimal bias-variance compromise between "estimate each alone (high variance)" and "pool all (high bias)" — the practical form of **Stein's paradox / empirical Bayes**, often beating both extremes. **Structure**: $y_k\sim \text{Lik}(\theta_k)$, $\theta_k\sim p(\theta|\phi)$, $\phi\sim$ hyperprior. Uses: A/B across segments, multi-region/store, recommendation cold-start, meta-analysis. Honest note: hyperprior choice and inferring $\tau$ (between-group variance) are sensitive with little data (the funnel problem → needs reparameterization).


**中文**：先看**概率编程长什么样**。以下是用 **PyMC**(本环境未安装,仅作展示)写我们这个层次打击率模型的样子——注意:你只"声明"了模型的每一层,**完全没写任何采样代码**,PyMC 会自动用 NUTS 采样:
**English**: First, **what probabilistic programming looks like**. Below is our hierarchical batting-average model written in **PyMC** (not installed here, shown for illustration) — note you only "declare" each layer of the model with **no sampling code at all**; PyMC auto-samples with NUTS:

```python
# ——— 概率编程示例(PyMC 风格, 本节实际用从零 MCMC 实现)/ PPL illustration ———
import pymc as pm
with pm.Model() as model:
    # 超先验:母分布的参数 / hyperpriors on the parent distribution
    mu    = pm.Beta("mu", 1, 1)                 # 全局平均打击率 / global mean rate
    kappa = pm.HalfNormal("kappa", 50)          # 组间集中度(方差反比)/ concentration
    # 各组参数从母分布抽出 / per-group params drawn from the parent
    theta = pm.Beta("theta", mu*kappa, (1-mu)*kappa, shape=K)
    # 似然:观测到的命中数 / likelihood
    hits  = pm.Binomial("hits", n=at_bats, p=theta, observed=y)
    trace = pm.sample(2000)                      # ← 一行, 自动 NUTS 推断! / one line, auto NUTS
```

**中文**：就这么几行。**你声明"数据是怎么生成的",PPL 负责"反推参数"**。下面我们**从零**(用 18.3 的 MCMC)实现同样的推断,以看清 PPL 背后到底在算什么。
**English**: Just those lines. **You declare "how the data is generated," the PPL handles "inferring the parameters."** Below we implement the same inference **from scratch** (with 18.3's MCMC) to see what the PPL computes under the hood.


In [ ]:

# ============================================================
# 数据:K 个球员的打击率, 上场次数差异很大 / K players' batting, very unequal at-bats
# 中文:真实打击率从一个"球员总体"里抽出。有人只上场几次(估计极不稳), 有人上百次。
# English: true averages drawn from a "player population." Some batted only a few times (noisy), some 100+.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
from scipy.special import betaln
rng=np.random.default_rng(2)
K=12
true_mu, kappa_true = 0.27, 90                                # 总体平均打击率 & 集中度 / population
theta_true = rng.beta(true_mu*kappa_true, (1-true_mu)*kappa_true, K)   # 每人真实打击率 / true rates
n = rng.integers(8, 120, K)                                  # 上场次数(差异大)/ at-bats (very unequal)
y = rng.binomial(n, theta_true)                              # 命中数 / hits observed

mle = y/n                                                    # ① 不合并:各自的命中率 / no pooling (MLE)
complete = y.sum()/n.sum()                                   # ② 完全合并:全局命中率 / complete pooling
print("上场次数 n:", n)
print("观测命中率(MLE) / observed rates:", np.round(mle,3))
print("小样本组的 MLE 极不稳 / small-n MLEs are wild:",
      f"最少上场 {n.min()} 次 → MLE {mle[np.argmin(n)]:.3f} (真值 {theta_true[np.argmin(n)]:.3f})")


**中文**：现在从零拟合**层次模型**(③ 部分合并)。模型:命中 $y_k\sim\text{Binomial}(n_k,\theta_k)$,各人打击率 $\theta_k\sim\text{Beta}(\mu\kappa,(1-\mu)\kappa)$ 从共享母分布抽出,超参数 $\mu$(全局均值)、$\kappa$(集中度)从数据估。我们用 **Metropolis-Hastings** 采样超参数(Beta-Binomial 的边际似然可解析,把 $\theta_k$ 积分掉),再得到每人 $\theta_k$ 的后验。
**English**: Now fit the **hierarchical model** (③ partial pooling) from scratch. Model: hits $y_k\sim\text{Binomial}(n_k,\theta_k)$, each rate $\theta_k\sim\text{Beta}(\mu\kappa,(1-\mu)\kappa)$ drawn from a shared parent, with hyperparameters $\mu$ (global mean) and $\kappa$ (concentration) estimated from data. We sample the hyperparameters with **Metropolis-Hastings** (the Beta-Binomial marginal likelihood is analytic, integrating out $\theta_k$), then get each $\theta_k$'s posterior.


In [ ]:

# ============================================================
# 从零拟合层次模型(MH 采样超参 + 解析后验θ)/ fit hierarchy from scratch
# ============================================================
def log_post(mu, kappa):                                     # 超参的对数后验(θ 已解析积分掉)/ hyperpost
    if not (0<mu<1) or kappa<=0: return -np.inf
    a, b = mu*kappa, (1-mu)*kappa
    ll = np.sum(betaln(a+y, b+n-y) - betaln(a, b))           # Beta-Binomial 边际似然 / marginal likelihood
    return ll + stats.norm.logpdf(np.log(kappa), 3, 2)       # 弱超先验 on log kappa / weak hyperprior

mu, lk = 0.25, np.log(50); cur=log_post(mu, np.exp(lk)); chain=[]
for i in range(30000):                                       # MH 采样超参数 mu, log kappa / MH sampling
    mu_n = mu + rng.normal(0,0.03); lk_n = lk + rng.normal(0,0.2)
    l = log_post(mu_n, np.exp(lk_n))
    if np.log(rng.random()) < l-cur: mu,lk,cur = mu_n,lk_n,l
    chain.append((mu, np.exp(lk)))
chain=np.array(chain[3000:])                                 # 丢 burn-in / discard burn-in
mu_hat, kappa_hat = chain.mean(0)
# 每人后验均值 = (a+y)/(a+b+n): 母分布把每人向全局均值收缩 / posterior mean shrinks toward global
a, b = mu_hat*kappa_hat, (1-mu_hat)*kappa_hat
partial = (a+y)/(a+b+n)                                      # ③ 部分合并估计 / partial-pooling estimate
print(f"估计的超参:全局均值 mu={mu_hat:.3f} (真值 {true_mu}), 集中度 kappa={kappa_hat:.0f}")
print("部分合并把每人向全局收缩, 小样本收缩最多 / partial pooling shrinks each toward global")


In [ ]:

# ============================================================
# 三种方法对比 + 可视化 / compare three approaches + visualization
# ============================================================
def rmse(est): return np.sqrt(np.mean((est-theta_true)**2))
r_no, r_comp, r_part = rmse(mle), rmse(np.full(K,complete)), rmse(partial)
print(f"{'方法/method':<26}{'RMSE vs 真实打击率':>18}")
print(f"{'① 不合并 no-pooling(MLE)':<26}{r_no:>18.4f}")
print(f"{'② 完全合并 complete-pool':<26}{r_comp:>18.4f}")
print(f"{'③ 部分合并 partial(hier)':<26}{r_part:>18.4f}  ← 最低!")

fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 收缩图:MLE → 部分合并 向全局均值收缩 / shrinkage plot
order=np.argsort(n)
for rank,k in enumerate(order):
    ax[0].plot([mle[k],partial[k]],[rank,rank],"-",color="gray",alpha=0.5)
    ax[0].plot(mle[k],rank,"o",color="#C44E52")
    ax[0].plot(partial[k],rank,"o",color="#4C72B0")
    ax[0].plot(theta_true[k],rank,"*",color="green",ms=9)
ax[0].axvline(complete,ls="--",color="k",label="全局均值 global mean")
ax[0].plot([],[],"o",color="#C44E52",label="MLE(不合并)"); ax[0].plot([],[],"o",color="#4C72B0",label="部分合并"); ax[0].plot([],[],"*",color="green",label="真实值")
ax[0].set_yticks(range(K)); ax[0].set_yticklabels([f"n={n[k]}" for k in order],fontsize=7)
ax[0].set_title("收缩:MLE→部分合并(小样本收缩最多)/ shrinkage"); ax[0].set_xlabel("打击率 batting avg"); ax[0].legend(fontsize=8)
# ② RMSE 对比 / RMSE bars
ax[1].bar(["不合并\nno-pool","完全合并\ncomplete","部分合并\npartial(hier)"],[r_no,r_comp,r_part],
          color=["#C44E52","#DD8452","#4C72B0"])
for i,v in enumerate([r_no,r_comp,r_part]): ax[1].text(i,v+0.001,f"{v:.4f}",ha="center")
ax[1].set_title("误差:部分合并两头都赢 / partial pooling wins"); ax[1].set_ylabel("RMSE vs 真实值")
plt.tight_layout(); plt.savefig("/tmp/bay05_viz.png",dpi=80); plt.show()
print("小样本组(顶部)的 MLE 离谱地远, 被大幅拉回全局均值附近; 大样本组(底部)几乎不动")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **部分合并(层次模型)两头都赢**:RMSE 上,层次模型(~0.020)**同时打败**了"各算各的"(~0.079,小样本噪声太大)和"全部合并"(~0.048,抹掉真实差异)。这就是**收缩(shrinkage)** 的威力——它在"信自己"和"信全局"之间,按每组数据量自动找最优折中。左图看得最清:**上场次数最少的球员(顶部),MLE 离真值最远,被大幅拉回全局均值;上场多的(底部)几乎不收缩**。这正是"数据少就多借力全局、数据多就信自己"。
2. **这是 Stein 悖论/经验贝叶斯的实践**:1955 年 Stein 证明了同时估计多个均值时,把各自估计向共同中心收缩**必然更准**——哪怕这些量毫不相关。层次模型是它的贝叶斯实现,在 A/B 分细分、多地区预估、推荐冷启动里天天创造价值。
3. **概率编程让这一切只需几行**:我们从零写了 MH 采样器,但用 PyMC/Stan 只需**声明模型结构**(如上方代码),推断全自动。这就是 PPL 的意义——**让你专注"怎么建模",把"怎么推断"交给编译器**。诚实局限:①超参 $\kappa$/$\tau$(组间方差)在**组少或组间差异小**时难估、后验对超先验敏感;②层次一深,后验几何会变"漏斗形"(funnel)让采样器卡住,需**非中心化重参数化**——这些正是 PPL 老手要处理的坑。

**English**:
1. **Partial pooling (hierarchical) wins both ways**: on RMSE, the hierarchical model (~0.020) **beats both** "estimate each alone" (~0.079, too noisy for small samples) and "pool everything" (~0.048, erases real differences). This is the power of **shrinkage** — it finds the optimal compromise between "trust yourself" and "trust the global," per group's sample size. The left plot shows it best: **the player with the fewest at-bats (top) has the most-off MLE and is pulled strongly back toward the global mean; the high-at-bat players (bottom) barely shrink**. Exactly "borrow from the global when data is scarce, trust yourself when it's plentiful."
2. **This is Stein's paradox / empirical Bayes in practice**: in 1955 Stein proved that when estimating several means jointly, shrinking each toward a common center is **provably more accurate** — even for unrelated quantities. The hierarchical model is its Bayesian implementation, creating value daily in A/B-by-segment, multi-region estimation, and recommendation cold-start.
3. **Probabilistic programming makes this a few lines**: we hand-wrote an MH sampler, but with PyMC/Stan you only **declare the model structure** (the code above) — inference is automatic. That is the point of PPLs — **focus on "how to model," let the compiler handle "how to infer."** Honest limits: ① the hyperparameters $\kappa$/$\tau$ (between-group variance) are hard to estimate and posterior-sensitive to the hyperprior when **groups are few or differences small**; ② deep hierarchies produce "funnel"-shaped posterior geometry that stalls samplers, needing **non-centered reparameterization** — exactly the pitfalls PPL veterans handle.

> 💼 **实战视角 / Practical angle**
> **中文**:层次模型是**工业界最实用的贝叶斯工具**之一:①**A/B 测试跨细分**(每个国家/设备/人群一个效应, 部分合并防小样本乱跳);②**多地区/多门店**销量/转化预估;③**推荐/广告冷启动**(新物品向品类均值收缩);④meta 分析(合并多个研究)。**工程**:用 **PyMC/NumPyro** 声明模型 + NUTS;数据少时**非中心化参数化**防漏斗;报告可信区间;检查 R-hat/ESS。面试金句:*"层次模型让各组参数共享一个母分布, 实现部分合并/收缩——小样本向全局借力、大样本信自己, 在偏差-方差上打败两个极端(Stein/经验贝叶斯); 概率编程(PyMC/Stan)让你只声明模型、自动推断。"*
> **English**: Hierarchical models are among **industry's most practical Bayesian tools**: ① **A/B testing across segments** (one effect per country/device/cohort, partial pooling tames small-sample noise); ② **multi-region/store** sales/conversion estimation; ③ **recommendation/ads cold-start** (new items shrink toward the category mean); ④ meta-analysis (combining studies). **Engineering**: declare the model in **PyMC/NumPyro** + NUTS; use **non-centered parameterization** to avoid funnels with little data; report credible intervals; check R-hat/ESS. Interview line: *"A hierarchical model shares a parent distribution across group parameters, giving partial pooling / shrinkage — small samples borrow from the global, large samples trust themselves, beating both extremes on bias-variance (Stein / empirical Bayes); probabilistic programming (PyMC/Stan) lets you just declare the model and auto-infer."*

---
### 小结 / Summary
- **中文**:概率编程=声明生成模型, 自动推断; 建模与推断解耦(PyMC/Stan/NumPyro)。
- **English**: Probabilistic programming = declare the generative model, auto-infer; decouple modeling from inference (PyMC/Stan/NumPyro).
- **中文**:层次模型=各组共享母分布→部分合并/收缩; 小样本向全局收缩多、大样本少, 两头都赢。
- **English**: Hierarchical model = groups share a parent → partial pooling / shrinkage; small samples shrink more, large less, winning both ways.
- **中文**:是 Stein 悖论/经验贝叶斯的实践, A/B 分细分、多地区、冷启动的利器; 注意超参敏感与漏斗问题。
- **English**: The practical form of Stein's paradox / empirical Bayes; a key tool for A/B-by-segment, multi-region, cold-start; mind hyperparameter sensitivity and the funnel problem.
